# SE3_model_train_validate

**Title:** An operational cloud based pipeline for near real time SWAT+ model  
**Author:** Md Zobayer Hossain Taki, University of Oulu  
**Repository:** [GitHub – cloud_swatplus_nrt_pipeline]

---

## Purpose
This notebook is **Step 3** of the four-part reproducible pipeline (SE1 → SE4).  
It uses **pySWATPlus** to calibrate, validate, and apply the SWAT+ model for daily streamflow prediction at the Oulanka River outlet.

## Structure

| Section | Cells | When to run |
|---------|-------|-------------|
| **Run Once** — Sensitivity, calibration, validation | Cells 3–6 | Once, or when re-calibrating |
| **Run Daily** — Near-real-time prediction | Cell 7 | Every day after SE1 → SE2 |

## pySWATPlus components used
1. **`TxtinoutReader`** — interact with the SWAT+ TxtInOut folder, configure runs, execute SWAT+
2. **`SensitivityAnalyzer`** — Sobol variance-based sensitivity indices (S1, ST)
3. **`Calibration`** — parameter optimisation via evolutionary algorithms (NSGA2)
4. **`PerformanceMetrics`** — NSE, KGE, RMSE, etc.

## Citation
Saló, J., Pal, D., & Llorente, O. (2025). swat-model/pySWATPlus. Zenodo. https://doi.org/10.5281/zenodo.14889319

## Inputs
- `TxtInOut_obs/` — observations-only SWAT+ model folder (from SE2)
- `TxtInOut_pred/` — observations + Harmonie forecast (from SE2)
- `Observed_data_standard_format_1.txt` — SYKE observed daily streamflow (from SE1 Part C)

## Outputs
| File | Cell | Description |
|------|------|-------------|
| `sensitivity_results.csv` | Cell 4 | Sobol S1, ST for all parameters |
| `optimized_parameters.csv` | Cell 5 | Best-fit parameter values (loaded or freshly calibrated) |
| `evaluation_metrics.csv` | Cell 6 | NSE, KGE, R², PBIAS for calibration and validation periods |
| `predictions_calval.csv` | Cell 6 | Daily observed vs simulated flow for plotting in SE4 |
| `predictions.csv` | Cell 7 | Daily streamflow, TN, TP for the prediction window |

## Reproducibility notes
- pySWATPlus `Calibration` and `SensitivityAnalyzer` use pymoo and SALib under the hood. **As of pySWATPlus latest release, neither class exposes a random seed parameter.** This means a fresh calibration may produce slightly different parameter values across runs.
- For reproducibility, `optimized_parameters.csv` is shipped with the repository — Cell 5 loads these values by default. Anyone cloning the repo gets identical predictions.
- To run a fresh calibration (e.g. with improved parameter bounds), set `CALIBRATION_MODE = "run_fresh"` in Cell 5.

## Cell 1 — Load dependencies

In [1]:
# Modify cell
from pathlib import Path
import os
import glob
import shutil
import datetime
import time
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pySWATPlus

# Show pySWATPlus and key package versions for reproducibility transparency
print(f"pySWATPlus : {pySWATPlus.__version__}")
print(f"pandas     : {pd.__version__}")
print(f"numpy      : {np.__version__}")

# Enable logging — shows generation/simulation progress during calibration
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

print("\nDependencies loaded.")

pySWATPlus : 1.3.0
pandas     : 3.0.0
numpy      : 2.3.5

Dependencies loaded.


## Cell 2 — Directory setup

All inputs are read from SE1/SE2 outputs. All new outputs are written outside the repository.

In [14]:
# Automatically generated cell — resolve external directories
notebook_dir  = Path.cwd()
project_root  = notebook_dir.parent
external_base = project_root.parent

# Processed data directory (outside repo)
proc_data_dir = external_base / "oulanka_swatplus_processeddata"
model_dir     = proc_data_dir / "model"
pred_dir      = proc_data_dir / "predictions"
for d in [model_dir, pred_dir]:
    d.mkdir(parents=True, exist_ok=True)

# ── USER SETTINGS ─────────────────────────────────────────────────────────────
BASE_DIR     = str(external_base / "oulanka_swatplus_rawdata" / "zenodo_data")
TARGET_UNIT  = 891         # SWAT+ channel unit ID for Oulanka main outlet
CHANNEL_NAME = 'cha1134'   # channel name in SWAT+ output  <-- update if needed

# Auto-detect TxtInOut_obs (created by SE2) using weather-sta.cli
# Sorted to ensure the latest dated folder (e.g., TxtInOut_Obs_Only_YYYY-MM-DD) is selected
obs_hits = sorted(list(Path(BASE_DIR).rglob("TxtInOut_Obs_Only_*/weather-sta.cli")))
if obs_hits:
    TXTINOUT_OBS = str(obs_hits[-1].parent)
else:
    TXTINOUT_OBS = os.path.join(BASE_DIR, "TxtInOut_obs")   # fallback

# Auto-detect TxtInOut_pred (created daily by SE2)
# Sorted to ensure the latest dated folder (e.g., TxtInOut_Full_Execution_YYYY-MM-DD) is selected
pred_hits = sorted(list(Path(BASE_DIR).rglob("TxtInOut_Full_Execution_*/weather-sta.cli")))
if pred_hits:
    TXTINOUT_PRED = str(pred_hits[-1].parent)
else:
    TXTINOUT_PRED = os.path.join(BASE_DIR, "TxtInOut_pred")   # fallback

# Observed flow file (from SE1 Part C)
syke_dir      = external_base / "oulanka_swatplus_rawdata" / "SYKE"
OBSERVED_FILE = str(syke_dir / "Observed_data_standard_format_1.txt")

print(f"TxtInOut_obs   : {TXTINOUT_OBS}  ",
      "[EXISTS]" if os.path.exists(TXTINOUT_OBS) else "[NOT FOUND — run SE2 first]")
print(f"TxtInOut_pred  : {TXTINOUT_PRED}  ",
      "[EXISTS]" if os.path.exists(TXTINOUT_PRED) else "[NOT FOUND — run SE2 first]")
print(f"Observed flow  : {OBSERVED_FILE}  ",
      "[EXISTS]" if os.path.exists(OBSERVED_FILE) else "[NOT FOUND — run SE1 Part C first]")

TxtInOut_obs   : /home/jovyan/oulanka_swatplus_rawdata/zenodo_data/TxtInOut_Obs_Only_2026-05-22   [EXISTS]
TxtInOut_pred  : /home/jovyan/oulanka_swatplus_rawdata/zenodo_data/TxtInOut_Full_Execution_2026-05-22   [EXISTS]
Observed flow  : /home/jovyan/oulanka_swatplus_rawdata/SYKE/Observed_data_standard_format_1.txt   [EXISTS]


In [12]:
# Automatically generated cell — resolve external directories
notebook_dir  = Path.cwd()
project_root  = notebook_dir.parent
external_base = project_root.parent

# Processed data directory (outside repo)
proc_data_dir = external_base / "oulanka_swatplus_processeddata"
model_dir     = proc_data_dir / "model"
pred_dir      = proc_data_dir / "predictions"
for d in [model_dir, pred_dir]:
    d.mkdir(parents=True, exist_ok=True)

# ── USER SETTINGS ─────────────────────────────────────────────────────────────
BASE_DIR     = str(external_base / "oulanka_swatplus_rawdata" / "zenodo_data")
TARGET_UNIT  = 891         # SWAT+ channel unit ID for Oulanka main outlet
CHANNEL_NAME = 'cha1134'   # channel name in SWAT+ output  <-- update if needed


# Auto-detect TxtInOut folders from SE2.
# Supports multiple naming conventions:
#   - SE2 v1 (date-stamped): TxtInOut_Obs_Only_YYYY-MM-DD, TxtInOut_Full_Execution_YYYY-MM-DD
#   - SE2 v2 (clean naming): TxtInOut_obs, TxtInOut_pred
# Picks the most recently modified folder if multiple exist.

def find_latest_folder(base_dir, patterns):
    """Return the path of the most recently modified folder matching any pattern."""
    candidates = []
    for pattern in patterns:
        candidates.extend(glob.glob(os.path.join(base_dir, pattern)))
    # Keep only folders that contain weather-sta.cli (valid TxtInOut folders)
    valid = [c for c in candidates if os.path.exists(os.path.join(c, "weather-sta.cli"))]
    if not valid:
        return None
    return max(valid, key=os.path.getmtime)

# Observation-only folder (from SE2)
TXTINOUT_OBS = find_latest_folder(BASE_DIR, [
    "TxtInOut_obs",                  # SE2 v2 naming
    "TxtInOut_Obs_Only_*"            # SE2 v1 date-stamped
])
if TXTINOUT_OBS is None:
    TXTINOUT_OBS = os.path.join(BASE_DIR, "TxtInOut_obs")   # fallback for error message

# Prediction folder with forecast (from SE2)
TXTINOUT_PRED = find_latest_folder(BASE_DIR, [
    "TxtInOut_pred",                 # SE2 v2 naming
    "TxtInOut_Full_Execution_*"      # SE2 v1 date-stamped
])
if TXTINOUT_PRED is None:
    TXTINOUT_PRED = os.path.join(BASE_DIR, "TxtInOut_pred")   # fallback for error message


# Observed flow file (from SE1 Part C)
syke_dir      = external_base / "oulanka_swatplus_rawdata" / "SYKE"
OBSERVED_FILE = str(syke_dir / "Observed_data_standard_format_1.txt")

print(f"TxtInOut_obs   : {TXTINOUT_OBS}  ",
      "[EXISTS]" if os.path.exists(TXTINOUT_OBS) else "[NOT FOUND — run SE2 first]")
print(f"TxtInOut_pred  : {TXTINOUT_PRED}  ",
      "[EXISTS]" if os.path.exists(TXTINOUT_PRED) else "[NOT FOUND — run SE2 first]")
print(f"Observed flow  : {OBSERVED_FILE}  ",
      "[EXISTS]" if os.path.exists(OBSERVED_FILE) else "[NOT FOUND — run SE1 Part C first]")

TxtInOut_obs   : /home/jovyan/oulanka_swatplus_rawdata/zenodo_data/TxtInOut_obs   [NOT FOUND — run SE2 first]
TxtInOut_pred  : /home/jovyan/oulanka_swatplus_rawdata/zenodo_data/TxtInOut_pred   [NOT FOUND — run SE2 first]
Observed flow  : /home/jovyan/oulanka_swatplus_rawdata/SYKE/Observed_data_standard_format_1.txt   [EXISTS]


---
# ═══════════════════════════════════════════════
# RUN ONCE — Sensitivity, Calibration, Validation
# (Cells 3 to 6)
# ═══════════════════════════════════════════════

## Cell 3 — Define parameter space

**Flow-focused calibration:** All 63 parameters from the original Finalization notebook are kept here for transparency, but Cell 4 (sensitivity analysis) identifies which ones actually matter for streamflow. Cell 5 then calibrates only the top sensitive parameters — this avoids the **equifinality problem** that hurt the original calibration (NSE = 0.22 with all 63 params).

**Why narrow bounds may be suboptimal:** The original calibration used very narrow bounds (e.g. `snomelt_tmp: 2.15–2.20`, range only 0.05). For improved calibration, consider wider literature-based bounds — see commented examples below.

In [10]:
# ==========================================
# CELL 3: PARAMETER SPACE
# 63 parameters covering all SWAT+ processes
# Bounds based on Arnold et al. (2012), Neitsch et al. (2011), and earlier TuneR work
# ==========================================

parameters = [
    # ===== Snow =====
    {'name': 'snomelt_tmp', 'change_type': 'absval', 'lower_bound': 2.15,   'upper_bound': 2.20},
    {'name': 'snofall_tmp', 'change_type': 'absval', 'lower_bound': 1.60,   'upper_bound': 3.00},
    # ===== Evapotranspiration =====
    {'name': 'esco',        'change_type': 'absval', 'lower_bound': 0.25,   'upper_bound': 0.30},
    {'name': 'epco',        'change_type': 'absval', 'lower_bound': 0.45,   'upper_bound': 0.55},
    {'name': 'awc',         'change_type': 'pctchg', 'lower_bound': -0.40,  'upper_bound': -0.25},
    {'name': 'canmx',       'change_type': 'pctchg', 'lower_bound': -0.60,  'upper_bound': -0.40},
    # ===== Surface Runoff =====
    {'name': 'cn2',         'change_type': 'pctchg', 'lower_bound': 0.60,   'upper_bound': 0.75},
    {'name': 'cn3_swf',     'change_type': 'absval', 'lower_bound': 0.21,   'upper_bound': 0.27},
    {'name': 'ovn',         'change_type': 'pctchg', 'lower_bound': -0.85,  'upper_bound': -0.80},
    {'name': 'surlag',      'change_type': 'absval', 'lower_bound': 10.0,   'upper_bound': 13.5},
    # ===== Lateral Flow =====
    {'name': 'lat_ttime',   'change_type': 'pctchg', 'lower_bound': 4.00,   'upper_bound': 4.50},
    {'name': 'lat_len',     'change_type': 'abschg', 'lower_bound': 40.0,   'upper_bound': 48.0},
    {'name': 'latq_co',     'change_type': 'absval', 'lower_bound': 0.40,   'upper_bound': 0.46},
    {'name': 'bd',          'change_type': 'pctchg', 'lower_bound': 0.53,   'upper_bound': 0.70},
    {'name': 'k',           'change_type': 'pctchg', 'lower_bound': -1.00,  'upper_bound': -0.94},
    # ===== Aquifer =====
    {'name': 'perco',       'change_type': 'absval', 'lower_bound': 0.01,   'upper_bound': 0.10},
    {'name': 'flo_min',     'change_type': 'abschg', 'lower_bound': 30.0,   'upper_bound': 37.0},
    {'name': 'revap_co',    'change_type': 'absval', 'lower_bound': 0.01,   'upper_bound': 0.07},
    {'name': 'revap_min',   'change_type': 'abschg', 'lower_bound': 2.50,   'upper_bound': 6.50},
    {'name': 'alpha',       'change_type': 'absval', 'lower_bound': 0.45,   'upper_bound': 0.60},
    {'name': 'sp_yld',      'change_type': 'absval', 'lower_bound': 0.13,   'upper_bound': 0.19},
    {'name': 'bf_max',      'change_type': 'absval', 'lower_bound': 0.70,   'upper_bound': 0.90},
    {'name': 'deep_seep',   'change_type': 'absval', 'lower_bound': 0.30,   'upper_bound': 0.34},
    # ===== Channel =====
    {'name': 'chn',         'change_type': 'absval', 'lower_bound': 0.20,   'upper_bound': 0.22},
    {'name': 'evol',        'change_type': 'absval', 'lower_bound': 2500.0, 'upper_bound': 3000.0},
    {'name': 'pvol',        'change_type': 'absval', 'lower_bound': 40.0,   'upper_bound': 60.0},
    # ===== Sediment =====
    {'name': 'cov',         'change_type': 'absval', 'lower_bound': 0.30,   'upper_bound': 0.50},
    {'name': 'ch_clay',     'change_type': 'absval', 'lower_bound': 85.0,   'upper_bound': 100.0},
    {'name': 'chs',         'change_type': 'pctchg', 'lower_bound': -1.00,  'upper_bound': -0.50},
    {'name': 'cherod',      'change_type': 'absval', 'lower_bound': -0.06,  'upper_bound': 0.06},
    {'name': 'prf',         'change_type': 'absval', 'lower_bound': 0.00,   'upper_bound': 2.00},
    {'name': 'lat_sed',     'change_type': 'absval', 'lower_bound': 2000.0, 'upper_bound': 4000.0},
    {'name': 'usle_p',      'change_type': 'pctchg', 'lower_bound': -1.00,  'upper_bound': 1.00},
    {'name': 'adj_pkr',     'change_type': 'absval', 'lower_bound': 1.50,   'upper_bound': 2.00},
    # ===== Nitrogen (kept for future TN calibration, not actively calibrated for flow) =====
    {'name': 'n_updis',     'change_type': 'absval', 'lower_bound': 40.0,   'upper_bound': 100.0},
    {'name': 'nperco',      'change_type': 'pctchg', 'lower_bound': -1.00,  'upper_bound': 1.00},
    {'name': 'sdnco',       'change_type': 'pctchg', 'lower_bound': -1.00,  'upper_bound': 1.00},
    {'name': 'cmn',         'change_type': 'pctchg', 'lower_bound': -1.00,  'upper_bound': 1.00},
    {'name': 'rsdco',       'change_type': 'absval', 'lower_bound': 0.02,   'upper_bound': 1.00},
    {'name': 'hlife_n',     'change_type': 'absval', 'lower_bound': 170.0,  'upper_bound': 178.0},
    {'name': 'no3_init',    'change_type': 'absval', 'lower_bound': 1.80,   'upper_bound': 1.88},
    {'name': 'lat_orgn',    'change_type': 'absval', 'lower_bound': 3.40,   'upper_bound': 4.80},
    {'name': 'rs4',         'change_type': 'absval', 'lower_bound': 0.001,  'upper_bound': 0.0016},
    {'name': 'bc3',         'change_type': 'absval', 'lower_bound': 0.10,   'upper_bound': 0.45},
    {'name': 'bc1',         'change_type': 'absval', 'lower_bound': 0.60,   'upper_bound': 0.95},
    {'name': 'bc2',         'change_type': 'absval', 'lower_bound': 1.90,   'upper_bound': 1.98},
    {'name': 'rs3',         'change_type': 'absval', 'lower_bound': 0.01,   'upper_bound': 0.04},
    {'name': 'erorgn',      'change_type': 'absval', 'lower_bound': 0.80,   'upper_bound': 0.98},
    {'name': 'cdn',         'change_type': 'absval', 'lower_bound': 2.55,   'upper_bound': 2.75},
    {'name': 'orgn',        'change_type': 'absval', 'lower_bound': 4.80,   'upper_bound': 4.98},
    {'name': 'nh3',         'change_type': 'absval', 'lower_bound': 0.82,   'upper_bound': 0.98},
    {'name': 'no2',         'change_type': 'absval', 'lower_bound': 0.15,   'upper_bound': 0.25},
    {'name': 'nsetlr1',     'change_type': 'absval', 'lower_bound': 0.30,   'upper_bound': 0.355},
    {'name': 'nsetlr2',     'change_type': 'absval', 'lower_bound': 2.00,   'upper_bound': 4.00},
    # ===== Phosphorus =====
    {'name': 'p_updis',     'change_type': 'absval', 'lower_bound': 85.0,   'upper_bound': 100.0},
    {'name': 'pperco',      'change_type': 'absval', 'lower_bound': 5.00,   'upper_bound': 10.22},
    {'name': 'phoskd',      'change_type': 'absval', 'lower_bound': 190.0,  'upper_bound': 197.0},
    {'name': 'psp',         'change_type': 'absval', 'lower_bound': 0.0182, 'upper_bound': 0.0185},
    {'name': 'erorgp',      'change_type': 'absval', 'lower_bound': 2.80,   'upper_bound': 3.04},
    {'name': 'usle_k',      'change_type': 'absval', 'lower_bound': 0.54,   'upper_bound': 0.645},
    {'name': 'lat_orgp',    'change_type': 'absval', 'lower_bound': 1.00,   'upper_bound': 1.50},
    {'name': 'rs5',         'change_type': 'absval', 'lower_bound': 0.06,   'upper_bound': 0.089},
    {'name': 'bc4',         'change_type': 'absval', 'lower_bound': 0.20,   'upper_bound': 0.38},
]

print(f"Total parameters defined: {len(parameters)}")

# ─── Suggested wider bounds for improved flow calibration (commented for reference) ───
# Replace the corresponding entries above with these wider, literature-based bounds
# if you choose CALIBRATION_MODE = 'run_fresh' in Cell 5.
#
# Snow (Neitsch et al. 2011):
#   snomelt_tmp : -5.0  to  5.0
#   snofall_tmp : -5.0  to  5.0
# ET:
#   esco        :  0.0  to  1.0
#   epco        :  0.0  to  1.0
#   awc (pct)   : -0.5  to  0.5
# Runoff:
#   cn2 (pct)   : -0.25 to  0.25
#   surlag      :  0.5  to  24.0

Total parameters defined: 63


## Cell 4 — Sobol sensitivity analysis

`pySWATPlus.SensitivityAnalyzer.simulation_and_indices()` computes Sobol indices via SALib:
- **S1** (first-order): contribution of parameter alone
- **ST** (total-order): contribution including all interactions

Parameters with high ST drive flow response and should be prioritised in calibration.  
Total simulations = `(2 × n_params + 2) × sample_number`. With 63 params × 4 samples ≈ 512 runs.

In [15]:
# ==========================================
# CELL 4: SOBOL SENSITIVITY ANALYSIS  [Run once]
# Uses pySWATPlus.SensitivityAnalyzer (SALib backend)
# ==========================================

# --- Set up sensitivity simulation directory ---
sens_sim_dir     = os.path.join(BASE_DIR, "sensitivity", "Model_Run_Sens")
sens_results_dir = os.path.join(BASE_DIR, "sensitivity", "Sensitivity_Results")

for folder in [sens_sim_dir, sens_results_dir]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder)

# Copy TxtInOut and configure for sensitivity period (1997-2002)
sens_base_reader = pySWATPlus.TxtinoutReader(tio_dir=TXTINOUT_OBS)
sens_base_reader.copy_required_files(sim_dir=sens_sim_dir)
sens_sim_reader  = pySWATPlus.TxtinoutReader(tio_dir=sens_sim_dir)

# Disable CSV print and enable monthly channel output
sens_sim_reader.disable_csv_print()
sens_sim_reader.enable_object_in_print_prt(
    obj='channel_sd', daily=False, monthly=True, yearly=True, avann=True
)

# Trial run to verify setup
sens_sim_reader.set_simulation_period(begin_date='01-Jan-1997', end_date='31-Dec-2002')
sens_sim_reader.set_warmup_year(warmup=2)
sens_sim_reader.run_swat()
print("Sensitivity setup verified with trial run.\n")

# --- Configure data extraction and metric ---
extract_data = {
    'channel_sd_mon.txt': {
        'has_units'   : True,
        'ref_day'     : 1,
        'apply_filter': {'name': [CHANNEL_NAME]}
    }
}
observe_data = {
    'channel_sd_mon.txt': {
        'obs_file'   : OBSERVED_FILE,
        'date_format': '%d-%m-%Y'
    }
}
metric_config = {
    'channel_sd_mon.txt': {
        'sim_col'  : 'flo_out',
        'obs_col'  : '7300100_q',
        'indicator': 'NSE'
    }
}

# --- Run Sobol sensitivity ---
if __name__ == '__main__':
    print(f"Running Sobol sensitivity: {len(parameters)} params × sample_number=4")
    print(f"Total SWAT+ runs: (2 × {len(parameters)} + 2) × 4 = {(2*len(parameters)+2)*4}\n")

    analyzer = pySWATPlus.SensitivityAnalyzer()
    sens_output = analyzer.simulation_and_indices(
        parameters    = parameters,
        sample_number = 4,
        sensim_dir    = sens_results_dir,
        txtinout_dir  = sens_sim_dir,
        extract_data  = extract_data,
        observe_data  = observe_data,
        metric_config = metric_config
    )

    # Save Sobol indices to CSV
    res     = sens_output[list(sens_output.keys())[0]]
    s1_vals = list(res['S1'].values()) if isinstance(res['S1'], dict) else list(res['S1'])
    st_vals = list(res['ST'].values()) if isinstance(res['ST'], dict) else list(res['ST'])
    sens_df = pd.DataFrame({
        'parameter': [p['name'] for p in parameters],
        'S1'       : s1_vals,
        'ST'       : st_vals
    }).sort_values('ST', ascending=False)

    sens_df.to_csv(pred_dir / "sensitivity_results.csv", index=False)
    print(f"Sensitivity results saved: {pred_dir / 'sensitivity_results.csv'}")
    print("\nTop 10 parameters by total-order sensitivity (ST):")
    print(sens_df.head(10).to_string(index=False))

TypeError: Expected exactly one executable file in the parent folder, but found none or multiple

## Cell 5 — Calibration (load existing or run fresh)

Two modes:

**`load_existing`** (default) — Reads `optimized_parameters.csv` shipped with the repository. Anyone cloning the repo gets identical predictions. Use this for reproducibility.

**`run_fresh`** — Runs `pySWATPlus.Calibration` (NSGA2, multi-objective). ~20–25 hours. Use this when calibrating with updated bounds or different objectives.

**Note on reproducibility:** pySWATPlus does not currently expose a `seed` parameter to its Calibration class, so two fresh runs may produce slightly different optimised values. To guarantee bit-identical predictions across users, ship the calibrated CSV with the repo (the default `load_existing` mode).

In [8]:
# ==========================================
# CELL 5: CALIBRATION  [Run once]
# Default: load shipped optimized_parameters.csv from the repo
# Alternative: run pySWATPlus.Calibration from scratch (~20-25 hrs)
# ==========================================

CALIBRATION_MODE = "load_existing"   # "load_existing" or "run_fresh"

opt_csv = model_dir / "optimized_parameters.csv"

# ────────────────────────────────────────────────────────────────
# MODE 1: LOAD EXISTING CALIBRATED VALUES
# ────────────────────────────────────────────────────────────────
if CALIBRATION_MODE == "load_existing":

    # In the published repository, optimized_parameters.csv is shipped under
    # `inputs/optimized_parameters.csv` for reproducibility. Copy it to model_dir.
    repo_csv = project_root / "inputs" / "optimized_parameters.csv"
    if repo_csv.exists() and not opt_csv.exists():
        shutil.copy(repo_csv, opt_csv)
        print(f"Copied shipped values from repo: {repo_csv}")

    if not opt_csv.exists():
        raise FileNotFoundError(
            f"optimized_parameters.csv not found at {opt_csv}\n"
            "Either:\n"
            "  1. Place the shipped CSV at inputs/optimized_parameters.csv in the repo, or\n"
            "  2. Set CALIBRATION_MODE = 'run_fresh' to calibrate from scratch."
        )

    opt_df = pd.read_csv(opt_csv)
    print(f"Loaded {len(opt_df)} calibrated parameters from: {opt_csv}")
    print(opt_df.head(10).to_string(index=False))

# ────────────────────────────────────────────────────────────────
# MODE 2: RUN FRESH CALIBRATION
# ────────────────────────────────────────────────────────────────
elif CALIBRATION_MODE == "run_fresh":

    # Set up an empty calibration directory (Calibration class requires empty folder)
    calibration_dir = os.path.join(BASE_DIR, "calibration", "NSGA2_run")
    if os.path.exists(calibration_dir):
        shutil.rmtree(calibration_dir)
    os.makedirs(calibration_dir)

    # Optionally restrict calibration to top-N sensitive parameters from Cell 4
    # This reduces equifinality and dramatically speeds up the run.
    USE_TOP_SENSITIVE  = True
    TOP_N              = 15
    if USE_TOP_SENSITIVE and (pred_dir / "sensitivity_results.csv").exists():
        sens_df       = pd.read_csv(pred_dir / "sensitivity_results.csv")
        top_names     = sens_df.sort_values('ST', ascending=False).head(TOP_N)['parameter'].tolist()
        calib_params  = [p for p in parameters if p['name'] in top_names]
        print(f"Calibrating top {TOP_N} sensitive parameters: {top_names}")
    else:
        calib_params  = parameters
        print(f"Calibrating all {len(parameters)} parameters")

    # Configure extract_data, observe_data, objective_config for flow
    # Both daily and monthly NSE — daily captures hydrograph shape; monthly captures volumes
    extract_data_cal = {
        'channel_sd_day.txt': {
            'has_units'   : True,
            'apply_filter': {'name': [CHANNEL_NAME]}
        },
        'channel_sd_mon.txt': {
            'has_units'   : True,
            'ref_day'     : 1,
            'apply_filter': {'name': [CHANNEL_NAME]}
        }
    }
    observe_data_cal = {
        'channel_sd_day.txt': {
            'obs_file'   : OBSERVED_FILE,
            'date_format': '%d-%m-%Y'
        },
        'channel_sd_mon.txt': {
            'obs_file'   : OBSERVED_FILE,
            'date_format': '%d-%m-%Y'
        }
    }
    objective_config_cal = {
        'channel_sd_day.txt': {
            'sim_col': 'flo_out', 'obs_col': '7300100_q', 'indicator': 'NSE'
        },
        'channel_sd_mon.txt': {
            'sim_col': 'flo_out', 'obs_col': '7300100_q', 'indicator': 'NSE'
        }
    }

    # Run NSGA2 calibration (no seed parameter — see reproducibility note in title)
    if __name__ == '__main__':
        print("Starting NSGA2 calibration — this will take 20-25 hours.")
        print("Keep the VRE session active.\n")

        calibration = pySWATPlus.Calibration(
            parameters       = calib_params,
            calsim_dir       = calibration_dir,
            txtinout_dir     = TXTINOUT_OBS,
            extract_data     = extract_data_cal,
            observe_data     = observe_data_cal,
            objective_config = objective_config_cal,
            algorithm        = 'NSGA2',
            n_gen            = 100,
            pop_size         = 120,
        )

        cal_output = calibration.parameter_optimization()
        print(f"\nCalibration finished. Runtime: {cal_output['time_sec']/3600:.2f} hours")

        # Take the first solution from the Pareto front
        if cal_output['variables'].ndim > 1:
            optimized_values = cal_output['variables'][0]
        else:
            optimized_values = cal_output['variables']

        # Save as standard CSV
        opt_df = pd.DataFrame({
            'parameter'      : [p['name'] for p in calib_params],
            'change_type'    : [p['change_type'] for p in calib_params],
            'lower_bound'    : [p['lower_bound'] for p in calib_params],
            'upper_bound'    : [p['upper_bound'] for p in calib_params],
            'optimized_value': optimized_values
        })
        opt_df.to_csv(opt_csv, index=False)
        print(f"Saved: {opt_csv}")
        print(opt_df.head(10).to_string(index=False))

else:
    raise ValueError(f"Unknown CALIBRATION_MODE: {CALIBRATION_MODE}")

Copied shipped values from repo: /home/jovyan/cloud_swatplus_nrt_pipeline/inputs/optimized_parameters.csv
Loaded 63 calibrated parameters from: /home/jovyan/oulanka_swatplus_processeddata/model/optimized_parameters.csv
  parameter change_type  lower_bound  upper_bound  optimized_value
snomelt_tmp      absval         2.15         2.20         2.199255
snofall_tmp      absval         1.60         3.00         2.957068
       esco      absval         0.25         0.30         0.250001
       epco      absval         0.45         0.55         0.468458
        awc      pctchg        -0.40        -0.25        -0.383858
      canmx      pctchg        -0.60        -0.40        -0.411666
        cn2      pctchg         0.60         0.75         0.665576
    cn3_swf      absval         0.21         0.27         0.268838
        ovn      pctchg        -0.85        -0.80        -0.822844
     surlag      absval        10.00        13.50        12.924331


## Cell 6 — Calibrated validation run + performance metrics

Run SWAT+ with the optimised parameters and compute NSE, KGE, R², PBIAS  
separately for the calibration and validation periods.

In [9]:
# ==========================================
# CELL 6: CALIBRATED VALIDATION RUN + METRICS  [Run once]
# ==========================================

# Calibration and validation period definitions
CAL_START, CAL_END     = '1999-01-01', '2006-12-31'
VAL_START, VAL_END     = '2007-01-01', '2010-12-31'
SIM_START, SIM_END     = '01-Jan-1997', '31-Dec-2010'
WARMUP_YEARS           = 2

# --- Run SWAT+ with calibrated parameters ---
cal_sim_dir = os.path.join(BASE_DIR, "model_run_calibrated")
if os.path.exists(cal_sim_dir):
    shutil.rmtree(cal_sim_dir)
os.makedirs(cal_sim_dir)

# Build parameter list in pySWATPlus run_swat() format
calibrated_params = [
    {'name': row['parameter'], 'change_type': row['change_type'], 'value': row['optimized_value']}
    for _, row in opt_df.iterrows()
]

# Configure and run
cal_base_reader = pySWATPlus.TxtinoutReader(tio_dir=TXTINOUT_OBS)
cal_base_reader.copy_required_files(sim_dir=cal_sim_dir)
cal_sim_reader  = pySWATPlus.TxtinoutReader(tio_dir=cal_sim_dir)
cal_sim_reader.enable_object_in_print_prt(
    obj='channel_sd', daily=True, monthly=True, yearly=True, avann=True
)

print(f"Running calibrated SWAT+ ({len(calibrated_params)} parameters applied)...")
t0 = time.time()
cal_sim_reader.run_swat(
    begin_date = SIM_START,
    end_date   = SIM_END,
    warmup     = WARMUP_YEARS,
    parameters = calibrated_params
)
mins, secs = divmod(time.time() - t0, 60)
print(f"Run complete. Runtime: {int(mins)} min {secs:.1f} s\n")

# --- Metric functions ---
def nse(obs, sim):   return 1 - np.sum((obs-sim)**2) / np.sum((obs-np.mean(obs))**2)
def kge(obs, sim):
    r = np.corrcoef(obs, sim)[0,1]
    b = np.mean(sim) / np.mean(obs)
    g = (np.std(sim)/np.mean(sim)) / (np.std(obs)/np.mean(obs))
    return 1 - np.sqrt((r-1)**2 + (b-1)**2 + (g-1)**2)
def r2(obs, sim):    return np.corrcoef(obs, sim)[0,1]**2
def pbias(obs, sim): return 100 * np.sum(obs-sim) / np.sum(obs)

# --- Load simulated and observed data, merge ---
df_sim_raw = pd.read_csv(os.path.join(cal_sim_dir, "channel_sd_day.txt"), sep=r'\s+', skiprows=[0,2])
df_sim_raw['date'] = pd.to_datetime(
    df_sim_raw[['yr','mon','day']].rename(columns={'yr':'year','mon':'month'})
)
df_sim = df_sim_raw[df_sim_raw['unit'] == TARGET_UNIT][['date','flo_out']].copy()

df_obs_flow         = pd.read_csv(OBSERVED_FILE)
df_obs_flow['date'] = pd.to_datetime(df_obs_flow['date'], dayfirst=True)
df_obs_flow.columns = ['date', 'obs_q']

merged = pd.merge(df_sim, df_obs_flow, on='date').dropna()
merged.to_csv(pred_dir / "predictions_calval.csv", index=False)

# --- Compute metrics for each period ---
metrics = {}
for name, (s, e) in [('calibration', (CAL_START, CAL_END)), ('validation', (VAL_START, VAL_END))]:
    sub = merged[(merged['date'] >= s) & (merged['date'] <= e)]
    if sub.empty:
        print(f"Warning: no merged data for {name} ({s} to {e})")
        continue
    o, s_ = sub['obs_q'].values, sub['flo_out'].values
    metrics[name] = {'NSE': nse(o,s_), 'KGE': kge(o,s_),
                     'R2' : r2(o,s_),  'PBIAS': pbias(o,s_)}

metrics_df = pd.DataFrame(metrics).T
metrics_df.to_csv(pred_dir / "evaluation_metrics.csv")
print("Performance metrics:")
print(metrics_df.round(3))
print(f"\nSaved: {pred_dir / 'evaluation_metrics.csv'}")

# --- Quick hydrograph plot for visual check ---
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(merged['date'], merged['obs_q'],   color='#0072B2', lw=1.2, label='Observed',  zorder=3)
ax.plot(merged['date'], merged['flo_out'], color='#D55E00', lw=1.0, label='Simulated', zorder=2)
ax.axvspan(pd.Timestamp(CAL_START), pd.Timestamp(CAL_END), alpha=0.08, color='#009E73', label='Calibration')
ax.axvspan(pd.Timestamp(VAL_START), pd.Timestamp(VAL_END), alpha=0.08, color='#E69F00', label='Validation')
ax.set_xlabel('Date'); ax.set_ylabel('Streamflow (m³/s)')
ax.set_title(
    f"Calibrated SWAT+ Hydrograph — Calib NSE={metrics_df.loc['calibration','NSE']:.3f}  "
    f"Valid NSE={metrics_df.loc['validation','NSE']:.3f}" if not metrics_df.empty else "Calibrated Hydrograph"
)
ax.legend(frameon=False, ncol=2); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

TypeError: Expected exactly one executable file in the parent folder, but found none or multiple

---
# ═══════════════════════════════════════════════
# RUN DAILY — Near-Real-Time Prediction
# (Cell 7)
# ═══════════════════════════════════════════════

## Cell 7 — Daily prediction run

Reads the daily-updated `TxtInOut_pred/` folder from SE2 (which contains observations + Harmonie NWP forecast)  
and applies the calibrated parameters from `optimized_parameters.csv`.  
Output `predictions.csv` is read by SE4 for figures and the dashboard.

In [ ]:
# ==========================================
# CELL 7: DAILY PREDICTION  [Run daily]
# ==========================================

# Verify pre-requisites
if not os.path.exists(TXTINOUT_PRED):
    raise FileNotFoundError(
        f"TxtInOut_pred folder not found: {TXTINOUT_PRED}\nRun SE2 first."
    )
if not opt_csv.exists():
    raise FileNotFoundError(
        f"optimized_parameters.csv not found: {opt_csv}\nRun Cell 5 first."
    )

# Auto-detect prediction end date from last line of .pcp file
pcp_files = glob.glob(os.path.join(TXTINOUT_PRED, "*.pcp"))
if pcp_files:
    with open(pcp_files[0]) as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]
    last_parts = lines[-1].split()
    last_date  = datetime.datetime(int(last_parts[0]), 1, 1) + datetime.timedelta(int(last_parts[1]) - 1)
    pred_end   = (last_date - datetime.timedelta(days=1)).strftime('%d-%b-%Y')
else:
    pred_end = (datetime.datetime.now() - datetime.timedelta(days=1)).strftime('%d-%b-%Y')

PRED_START = '01-Jan-2015'   # adjust if needed
print(f"Prediction period: {PRED_START} → {pred_end}")

# Set up timestamped simulation directory
time_str    = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
predict_dir = os.path.join(BASE_DIR, f"prediction_run_{time_str}")
os.makedirs(predict_dir)

# Load calibrated parameters
opt_df  = pd.read_csv(opt_csv)
params  = [
    {'name': row['parameter'], 'change_type': row['change_type'], 'value': row['optimized_value']}
    for _, row in opt_df.iterrows()
]

# Configure and run SWAT+
pred_reader = pySWATPlus.TxtinoutReader(tio_dir=TXTINOUT_PRED)
pred_reader.copy_required_files(sim_dir=predict_dir)
sim_reader  = pySWATPlus.TxtinoutReader(tio_dir=predict_dir)
sim_reader.enable_object_in_print_prt(
    obj='channel_sd', daily=True, monthly=True, yearly=True, avann=True
)

print("\nRunning prediction...")
t0 = time.time()
sim_reader.run_swat(
    begin_date = PRED_START,
    end_date   = pred_end,
    warmup     = 2,
    parameters = params
)
mins, secs = divmod(time.time() - t0, 60)
print(f"Prediction complete. Runtime: {int(mins)} min {secs:.1f} s")

# --- Extract flow, TN, TP from output ---
out_file = os.path.join(predict_dir, "channel_sd_day.txt")
df_out   = pd.read_csv(out_file, sep=r'\s+', skiprows=1)

# Identify unit column (varies between SWAT+ versions)
unit_col = 'unit' if 'unit' in df_out.columns else ('gis_id' if 'gis_id' in df_out.columns else df_out.columns[4])
df_outlet           = df_out[df_out[unit_col] == TARGET_UNIT].copy()
df_outlet['Date']   = pd.to_datetime(
    df_outlet[['yr','mon','day']].rename(columns={'yr':'year','mon':'month'})
)

# Total Nitrogen / Total Phosphorus = sum of constituent fractions
n_cols              = [c for c in ['orgn_out','no3_out','no2_out','nh3_out'] if c in df_outlet.columns]
p_cols              = [c for c in ['orgp_out','solp_out'] if c in df_outlet.columns]
df_outlet['TN']     = df_outlet[n_cols].sum(axis=1) if n_cols else np.nan
df_outlet['TP']     = df_outlet[p_cols].sum(axis=1) if p_cols else np.nan

# Save predictions for SE4
df_outlet[['Date','flo_out','TN','TP']].to_csv(pred_dir / "predictions.csv", index=False)

print(f"\nPredictions saved: {pred_dir / 'predictions.csv'}")
print(f"Period           : {df_outlet['Date'].min().date()} to {df_outlet['Date'].max().date()}")
print("\nLast 3 days (forecast):")
print(df_outlet.tail(3)[['Date','flo_out','TN','TP']].to_string(index=False))
print("\nSE3 complete. Run SE4 to generate figures and the forecast dashboard.")

---
## Summary

**First time / re-calibration:** Run all cells (1 → 7) in order. Cell 5 takes ~25 hours in `run_fresh` mode, or seconds in `load_existing` mode.

**Daily prediction:** Run Cells 1, 2, then jump to Cell 7. (Cell 5 will load the existing optimised parameters automatically.)

**Files written:**
- `sensitivity_results.csv` — Sobol S1, ST (Cell 4)
- `optimized_parameters.csv` — calibrated parameter values (Cell 5)
- `evaluation_metrics.csv`, `predictions_calval.csv` — performance evaluation (Cell 6)
- `predictions.csv` — daily flow, TN, TP for the prediction window (Cell 7)

---
## Need help?
https://github.com/orgs/DigitalWaters-fi/discussions — Tag **#modelling**